## Transfer Learning using MNIST data
To illustrate the power and concept of transfer learning, we will train a CNN on just the digits 5,6,7,8,9.  Then we will train just the last layer(s) of the network on the digits 0,1,2,3,4 and see how well the features learned on 5-9 help with classifying 0-4.



In [50]:


import datetime
import keras
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras import backend as K

In [51]:
#used to help some of the timing functions
now = datetime.datetime.now

In [52]:
# set some parameters
batch_size = 128
num_classes = 5
epochs = 5

In [53]:
# set some more parameters
img_rows, img_cols = 28, 28
filters = 32
pool_size = 2
kernel_size = 3

In [54]:
input_shape= 28, 28, 1

In [72]:
## To simplify things, write a function to include all the training steps
## As input, function takes a model, training set, test set, and the number of classes
## Inside the model object will be the state about which layers we are freezing and which we are training
#Reshape the data
#Normalize the data
# One hot encode the targert label
# Compile the model
# Train the model on the training data
# Evaluate the model on the testing data

def train_model(model, train, test, num_classes):
    X_train= train[0].reshape((train[0].shape[0],) + input_shape)
    X_test= test[0].reshape((test[0].shape[0],) + input_shape)
    X_train = X_train.astype('float32')
    X_test= X_test.astype('float32')
    X_train /=255
    X_test /=255
    print('X_train shape:', X_train.shape)
    print(X_train.shape[0], 'train samples')
    print(X_test.shape[0], 'test samples')

    y_train = keras.utils.to_categorical(train[1], num_classes)
    y_test = keras.utils.to_categorical (test[1], num_classes)

    model.compile(loss='categorical_crossentropy',
                  optimizer='adadelta',
                  metrics=['accuracy'])

    t= now()
    model.fit(X_train, y_train,
               batch_size=batch_size,
               epochs=epochs,
               verbose=1,
               validation_data=(X_test, y_test))
    print('Training time: %s' %(now() -t))

    score=model.evaluate(X_test, y_test, verbose=0)
    print('Test score:', score[0])
    print('Test accuracy:', score[1])

In [73]:
# Load the Mnist data and split between train and test sets
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# create two datasets: one with digits below 5 and one with 5 and above
X_train_lt5=X_train[y_train <5]
y_train_lt5=y_train[y_train < 5]
X_test_lt5=X_test[y_test<5]
y_test_lt5=y_test[y_test<5]
X_train_gte5=X_train[y_train>=5]
y_train_gte5=y_train[y_train>=5]-5
X_test_gte5=X_test[y_test>=5]
y_test_gte5=y_test[y_test>=5]-5


In [74]:
# Define the "feature" layers. Add 2 convolution layer with max pool layer. At the end, add dropout layer with 0.25% probability and end with the flatten layer. These are the early layers that we expect will "transfer"
# to a new problem.  We will freeze these layers during the fine-tuning process

feature_layers = [
    Conv2D(filters, kernel_size,
           padding='valid',
           input_shape=input_shape),
    Activation('relu'),
    Conv2D(filters,kernel_size),
    Activation('relu'),
    MaxPooling2D(pool_size=pool_size),
    Dropout(0.25),
    Flatten(),
   
]

In [75]:
# Define the "classification" layers. Add Dense layer with 128 nodes and the output dense layer. These are the later layers that predict the specific classes from the features
# learned by the feature layers.  This is the part of the model that needs to be re-trained for a new problem

classification_layers = [
    Dense(128),
    Activation('relu'),
    Dropout(0.5),
    Dense(num_classes),
    Activation('softmax')

]

In [76]:
# Create the model by combining the two sets of layers:
model = Sequential(feature_layers + classification_layers)

In [77]:
# print model summary
model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_14 (Conv2D)          (None, 26, 26, 32)        320       
                                                                 
 activation_28 (Activation)  (None, 26, 26, 32)        0         
                                                                 
 conv2d_15 (Conv2D)          (None, 24, 24, 32)        9248      
                                                                 
 activation_29 (Activation)  (None, 24, 24, 32)        0         
                                                                 
 max_pooling2d_7 (MaxPooling  (None, 12, 12, 32)       0         
 2D)                                                             
                                                                 
 dropout_14 (Dropout)        (None, 12, 12, 32)        0         
                                                      

In [78]:
# Now, let's train our model on the digits 5,6,7,8,9
train_model(model,
            (X_train_gte5, y_train_gte5),
            (X_test_gte5, y_test_gte5), num_classes)


X_train shape: (29404, 28, 28, 1)
29404 train samples
4861 test samples
Epoch 1/5
230/230 [==============================] - 8s 35ms/step - loss: 1.6039 - accuracy: 0.2296 - val_loss: 1.5780 - val_accuracy: 0.3191
Epoch 2/5
230/230 [==============================] - 9s 39ms/step - loss: 1.5698 - accuracy: 0.3041 - val_loss: 1.5410 - val_accuracy: 0.5188
Epoch 3/5
230/230 [==============================] - 8s 36ms/step - loss: 1.5360 - accuracy: 0.3779 - val_loss: 1.5020 - val_accuracy: 0.6838
Epoch 4/5
230/230 [==============================] - 8s 35ms/step - loss: 1.4977 - accuracy: 0.4550 - val_loss: 1.4589 - val_accuracy: 0.7616
Epoch 5/5
230/230 [==============================] - 8s 35ms/step - loss: 1.4581 - accuracy: 0.5106 - val_loss: 1.4109 - val_accuracy: 0.7930
Training time: 0:00:41.522526
Test score: 1.410857915878296
Test accuracy: 0.7930467128753662


### Freezing Layers
Keras allows layers to be "frozen" during the training process.  That is, some layers would have their weights updated during the training process, while others would not.  This is a core part of transfer learning, the ability to train just the last one or several layers.

Note also, that a lot of the training time is spent "back-propagating" the gradients back to the first layer.  Therefore, if we only need to compute the gradients back a small number of layers, the training time is much quicker per iteration.  This is in addition to the savings gained by being able to train on a smaller data set.

In [88]:
# Freeze only the feature layers
for l in feature_layers:
    l.trainable=False



Observe below the differences between the number of *total params*, *trainable params*, and *non-trainable params*.

In [89]:
# print model summary
model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_14 (Conv2D)          (None, 26, 26, 32)        320       
                                                                 
 activation_28 (Activation)  (None, 26, 26, 32)        0         
                                                                 
 conv2d_15 (Conv2D)          (None, 24, 24, 32)        9248      
                                                                 
 activation_29 (Activation)  (None, 24, 24, 32)        0         
                                                                 
 max_pooling2d_7 (MaxPooling  (None, 12, 12, 32)       0         
 2D)                                                             
                                                                 
 dropout_14 (Dropout)        (None, 12, 12, 32)        0         
                                                      

In [90]:
# Now, let's train our model on the digits 0,1,2,3,4
train_model(model,
            (X_train_lt5, y_train_lt5),
            (X_test_lt5, y_test_lt5), num_classes)


X_train shape: (30596, 28, 28, 1)
30596 train samples
5139 test samples
Epoch 1/5
240/240 [==============================] - 4s 14ms/step - loss: 1.5497 - accuracy: 0.3357 - val_loss: 1.5040 - val_accuracy: 0.5487
Epoch 2/5
240/240 [==============================] - 3s 14ms/step - loss: 1.4949 - accuracy: 0.4224 - val_loss: 1.4484 - val_accuracy: 0.6295
Epoch 3/5
240/240 [==============================] - 3s 14ms/step - loss: 1.4430 - accuracy: 0.4947 - val_loss: 1.3932 - val_accuracy: 0.6906
Epoch 4/5
240/240 [==============================] - 4s 17ms/step - loss: 1.3938 - accuracy: 0.5554 - val_loss: 1.3380 - val_accuracy: 0.7552
Epoch 5/5
240/240 [==============================] - 4s 15ms/step - loss: 1.3437 - accuracy: 0.6140 - val_loss: 1.2827 - val_accuracy: 0.8114
Training time: 0:00:18.073444
Test score: 1.2826762199401855
Test accuracy: 0.8114418983459473


Note that after a single epoch, we are already achieving results on classifying 0-4 that are comparable to those achieved on 5-9 after 5 full epochs.  This despite the fact the we are only "fine-tuning" the last layer of the network, and all the early layers have never seen what the digits 0-4 look like.

Also, note that even though nearly all (590K/600K) of the *parameters* were trainable, the training time per epoch was still much reduced.  This is because the unfrozen part of the network was very shallow, making backpropagation faster. 